# Update download statistics

This notebook queries the public PyPI download dataset in Google BigQuery and merges every completed month after the latest month in `downloads.csv`.

Before running it on a new machine, follow [Download statistics setup](README.md) to install the Google Cloud CLI, configure Application Default Credentials, install the `stats` dependencies, and select the correct Jupyter kernel.

Run the cells in order. The final cell writes the merged and deduplicated records back to `downloads.csv`.

In [1]:
import csv
import os

from pathlib import Path

from google.auth import default as google_auth_default
from google.cloud import bigquery

## Authentication

Complete the one-time [download statistics setup](README.md) before running this notebook. The next cell loads Application Default Credentials and uses `GOOGLE_CLOUD_PROJECT` when set; otherwise it uses the project detected from those credentials.

In [2]:
credentials, detected_project = google_auth_default()
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT") or detected_project or credentials.quota_project_id

if not project_id:
    raise RuntimeError(
        "No Google Cloud project was detected. Set GOOGLE_CLOUD_PROJECT or run "
        "'gcloud auth application-default set-quota-project micropython-stubs'."
    )

client = bigquery.Client(project=project_id, credentials=credentials)
print(f"Authenticated with {type(credentials).__name__}; query project: {project_id}")

Authenticated with Credentials; query project: micropython-stubs


In [3]:
def get_monthly_stats(year, month):
    query = f"""
        SELECT
          COUNT(*) AS downloads,
          REGEXP_EXTRACT(file.project, r".*?-(.*?)-(?:.*-)?stubs") AS port,
          REGEXP_EXTRACT(file.project, r".*?-.*?-(?:(.*)-)?stubs") AS board,
          REGEXP_EXTRACT(file.version, r"(.*).post") AS version,
          DATE({year},{month},1) AS report_date,
          file.project AS project,
          file.version AS version_full
        FROM
          `bigquery-public-data.pypi.file_downloads`
        WHERE
          file.project LIKE 'micropython-%-stubs'
          AND DATE(timestamp) >= DATE({year},{month},1)
          AND DATE(timestamp) < DATE_ADD(DATE({year},{month},1), INTERVAL 1 MONTH)
          AND details.installer.name <> 'bandersnatch'
        GROUP BY
          port,
          board,
          version,
          project,
          version_full
        ORDER BY
          downloads DESC
        """
    return client.query(query).result()

In [4]:
import datetime

stats_list = []
current_month = datetime.date.today().replace(day=1)

print(f"Collecting completed months before {current_month:%Y-%m}")

In [5]:
csv_file = Path("downloads.csv")
if not csv_file.exists():
    csv_file = Path("statistics/downloads.csv")
if not csv_file.exists():
    raise FileNotFoundError("Run this notebook from its folder or the repository root.")

with csv_file.open("r", encoding="utf-8", newline="") as input_file:
    current_stats_list = list(csv.DictReader(input_file))

current_stats_list.sort(key=lambda record: record["report_date"])
last_report_month = datetime.date.fromisoformat(current_stats_list[-1]["report_date"])
next_report_month = (last_report_month.replace(day=28) + datetime.timedelta(days=4)).replace(day=1)

print(f"Latest stored month: {last_report_month:%Y-%m}")
print(f"Next month to collect: {next_report_month:%Y-%m}")

Latest stored month: 2024-07
Next month to collect: 2024-08


In [6]:
report_month = next_report_month

while report_month < current_month:
    print(f"Processing {report_month:%Y-%m} ... ", end="", flush=True)

    results = get_monthly_stats(report_month.year, report_month.month)
    print(f"Retrieved {results.total_rows} download summaries")
    stats_list.extend(dict(row) for row in results)

    report_month = (report_month.replace(day=28) + datetime.timedelta(days=4)).replace(day=1)

print(f"Collected {len(stats_list)} new rows")

Processing 2024-08 ... Retrieved 221 download summaries
Processing 2024-09 ... Retrieved 236 download summaries
Processing 2024-10 ... Retrieved 238 download summaries
Processing 2024-11 ... Retrieved 238 download summaries
Processing 2024-12 ... Retrieved 238 download summaries
Processing 2025-01 ... Retrieved 238 download summaries
Processing 2025-02 ... Retrieved 256 download summaries
Processing 2025-03 ... Retrieved 273 download summaries
Processing 2025-04 ... Retrieved 291 download summaries
Processing 2025-05 ... Retrieved 277 download summaries
Processing 2025-06 ... Retrieved 330 download summaries
Processing 2025-07 ... Retrieved 328 download summaries
Processing 2025-08 ... Retrieved 331 download summaries
Processing 2025-09 ... Retrieved 351 download summaries
Processing 2025-10 ... Retrieved 359 download summaries
Processing 2025-11 ... Retrieved 322 download summaries
Processing 2025-12 ... Retrieved 380 download summaries
Processing 2026-01 ... Retrieved 380 download su

In [7]:
full_list = current_stats_list + stats_list

In [8]:
# avoid double counts
unique = {}
for rec in full_list:
    key = f"{rec['report_date']}-{rec['project']}-{rec['version_full']}"
    unique[key] = rec

full_list = list(unique.values())

In [9]:
import datetime

print("sorting")

# Convert "report_date" strings to datetime.date objects
for rec in full_list:
    if isinstance(rec["report_date"], str):
        rec["report_date"] = datetime.datetime.strptime(rec["report_date"], "%Y-%m-%d").date()

full_list.sort(key=lambda x: (x["report_date"], x["project"], x["version_full"]))

sorting


In [10]:
print("writing to csv")
keys = full_list[0].keys()
# Convert "report_date" strings to datetime objects

with open(csv_file, "w", newline="") as output_file:
    dict_writer = csv.DictWriter(output_file, keys, lineterminator=os.linesep)
    dict_writer.writeheader()

    dict_writer.writerows(full_list)

writing to csv
